# Analýza náhrad výdavkov

Tento notebook demonštruje, ako vytvoriť agentov, ktorí používajú pluginy na spracovanie cestovných výdavkov z miestnych obrázkov účtov, vytvárajú e-mail so žiadosťou o náhradu výdavkov a vizualizujú údaje o výdavkoch pomocou koláčového grafu. Agenti dynamicky vyberajú funkcie na základe kontextu úlohy.

Kroky:
1. OCR Agent spracuje miestny obrázok účtu a extrahuje údaje o cestovných výdavkoch.
2. Email Agent generuje e-mail so žiadosťou o náhradu výdavkov.

### Príklad scenára cestovných výdavkov:
Predstavte si, že ste zamestnanec cestujúci na pracovné stretnutie do iného mesta. Vaša spoločnosť má politiku preplácania všetkých rozumných výdavkov súvisiacich s cestovaním. Tu je rozpis potenciálnych cestovných výdavkov:
- Dopravné:
Letecká doprava na spiatočnú cestu z vášho domovského mesta do cieľového mesta.
Taxi alebo služby zdieľanej dopravy na letisko a z letiska.
Miestna doprava v cieľovom meste (napríklad verejná doprava, prenájom áut alebo taxíky).

- Ubytovanie:
Ubytovanie v hoteli na tri noci v stredne drahom biznis hoteli blízko miesta stretnutia.

- Stravovanie:
Denný stravný príspevok na raňajky, obed a večeru, podľa firemnej politiky per diem.

- Rôzne výdavky:
Poplatky za parkovanie na letisku.
Poplatky za internetové pripojenie v hoteli.
Tipy alebo malé servisné poplatky.

- Dokumentácia:
Predložíte všetky účty (letenky, taxíky, hotel, jedlo atď.) a vyplnenú správu o výdavkoch na preplatenie.


## Import potrebných knižníc

Importujte potrebné knižnice a moduly pre tento notebook.


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

 ## Definovať modely výdavkov

 Vytvorte Pydantic model pre jednotlivé výdavky a triedu ExpenseFormatter na konverziu používateľského dopytu do štruktúrovaných údajov o výdavkoch.

 Každý výdavok bude reprezentovaný v tvare:
 `{'date': '07-Mar-2025', 'description': 'let na destináciu', 'amount': 675.99, 'category': 'Doprava'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Defining Tools - Generating the Email

Vytvorte funkciu nástroja na generovanie e-mailu na podanie žiadosti o refundáciu výdavkov.
- Tento nástroj používa dekorátor `@tool` z Microsoft Agent Framework.
- Vypočíta celkovú sumu výdavkov a naformátuje podrobnosti do tela e-mailu.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Nástroj na extrakciu cestovných výdavkov z obrázkov účteniek

Vytvorte nástrojovú funkciu na extrakciu cestovných výdavkov z obrázkov účteniek.
- Tento nástroj používa dekorátor `@tool` z Microsoft Agent Framework.
- Číta obrázok účtenky, kóduje ho do base64 a vracia data URI na analýzu agentom.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Spracovanie výdavkov

Definujte agentov a prepojte ich do sekvenčného pracovného postupu pomocou `WorkflowBuilder`.
- OCR agent extrahuje štruktúrované údaje o výdavkoch z obrázka účtenky pomocou nástroja `load_receipt_image`.
- Email agent použije extrahované údaje a vytvorí profesionálny e-mail na preukázanie výdavkov pomocou nástroja `generate_expense_email`.
- `WorkflowBuilder` s `add_edge` vytvorí sekvenčný reťazec: OCR agent → Email agent.


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Hlavná funkcia

Vytvorte sekvenčný pracovný tok a spustite ho na spracovanie obrázka účtenky a vygenerovanie e-mailu s preukazom výdavkov.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zrieknutie sa zodpovednosti**:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Hoci sa snažíme o presnosť, prosím, majte na pamäti, že automatizované preklady môžu obsahovať chyby alebo nepresnosti. Pôvodný dokument v jeho rodnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za akékoľvek nedorozumenia alebo nesprávne výklady vyplývajúce z používania tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
